# fase_5 - script_hanif Migration

This notebook handles migration of database from old DB to new DB for fase 5.

**Purpose**: Migrasi data Rapor dan Penilaian dengan Mapping Kolom Spesifik

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
import re
from datetime import datetime
import pickle
import json
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [2]:
config = get_db_config()
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f"Connected to {config['db_old']['database']} and {config['db_new']['database']}")

Connected to dataleap_v5_example and dataleap_v5_migration


## 2. Ambil Data dari DB Lama

In [3]:
hanif_tables_map = [
    ('format_rapor', 'rapor_format'),
    ('format_rapor_detil', 'rapor_format_sub'),
    ('format_rapor_rumus', 'rapor_format_formula'),
    ('format_rapor_detil_rumus', 'rapor_format_formula_sub'),
    ('format_raport_level', 'rapor_level_config'),
    ('rapor', 'rapor_siswa'),
    ('file_rapor_siswa', 'rapor_siswa_file'),
    ('history_rapor', 'rapor_lacak')
]

raw_data = {}
for old_t, new_t in hanif_tables_map:
    try:
        cursor_old.execute(f"SELECT * FROM `{old_t}`")
        raw_data[old_t] = cursor_old.fetchall()
        print(f"✅ {old_t} loaded: {len(raw_data[old_t])} records")
    except Exception as e:
        print(f"❌ ERROR loading {old_t}: {e}")

✅ format_rapor loaded: 45 records
✅ format_rapor_detil loaded: 129 records
✅ format_rapor_rumus loaded: 3 records
✅ format_rapor_detil_rumus loaded: 1650 records
✅ format_raport_level loaded: 348 records
✅ rapor loaded: 22837 records
✅ file_rapor_siswa loaded: 1506 records
✅ history_rapor loaded: 1366 records


## 3. Transform Data (Mapping Berdasarkan hanif_mapping.md)

In [4]:
transformed_dfs = {}

# --- TRANSFORMATION ---

# 1. format_rapor -> rapor_format
if 'format_rapor' in raw_data:
    df = pd.DataFrame(raw_data['format_rapor'])
    mapping = {
        'idformat_rapor': 'id_rapor_format',
        'idpendkursus': 'id_kursus', 'title': 'judul_rapor'
    }
    transformed_dfs['rapor_format'] = df.rename(columns=mapping)[list(mapping.values())]

# 2. format_rapor_detil -> rapor_format_sub
if 'format_rapor_detil' in raw_data:
    df = pd.DataFrame(raw_data['format_rapor_detil'])
    mapping = {
        'idformat_rd': 'id_rapor_format_sub',
        'idformat_rapor': 'id_rapor_format', 'subtitle': 'sub_judul_rapor'
    }
    transformed_dfs['rapor_format_sub'] = df.rename(columns=mapping)[list(mapping.values())]

# 3. format_rapor_rumus -> rapor_format_formula
if 'format_rapor_rumus' in raw_data:
    df = pd.DataFrame(raw_data['format_rapor_rumus'])
    mapping = {
        'idfrr': 'id_rapor_format_formula',
        'idformat_rapor': 'id_rapor_format', 'param_operator': 'logika_operator'
    }
    transformed_dfs['rapor_format_formula'] = df.rename(columns=mapping)[list(mapping.values())]

# 4. format_rapor_detil_rumus -> rapor_format_formula_sub
if 'format_rapor_detil_rumus' in raw_data:
    df = pd.DataFrame(raw_data['format_rapor_detil_rumus'])
    mapping = {
        'idfrdr': 'id_rapor_format_formula_sub',
        'idformat_rd': 'id_rapor_format_sub', 'param_operator': 'logika_operator',
        'idlevel': 'id_level'
    }
    transformed_dfs['rapor_format_formula_sub'] = df.rename(columns=mapping)[list(mapping.values())]

# 5. format_raport_level -> rapor_level_config
if 'format_raport_level' in raw_data:
    df = pd.DataFrame(raw_data['format_raport_level'])
    mapping = {
        'idformat_rl': 'id_rapor_level_config', 'idlevel': 'id_level',
        'idpendkursus': 'id_kursus', 'idformat_rapor': 'id_rapor_format'
    }
    transformed_dfs['rapor_level_config'] = df.rename(columns=mapping)[list(mapping.values())]

# 6. rapor_sub_level (Tabel Baru)
transformed_dfs['rapor_sub_level'] = pd.DataFrame(columns=['id_rapor_sub_level', 'id_rapor_format_sub', 'id_level'])

# 7. rapor -> rapor_siswa
if 'rapor' in raw_data:
    df = pd.DataFrame(raw_data['rapor'])
    mapping = {
        'idrapor': 'id_rapor_siswa', 'idjadwal': 'id_jadwal', 'idsiswa': 'id_siswa',
        'tanggal': 'tanggal_input', 'idp_nilai': 'id_parameter_nilai', 'nilai': 'final_result'
    }
    # Note: tanggal direct mapping
    transformed_dfs['rapor_siswa'] = df.rename(columns=mapping)[list(mapping.values())]

# 8. file_rapor_siswa -> rapor_siswa_file
if 'file_rapor_siswa' in raw_data and 'rapor_siswa' in transformed_dfs:
    df = pd.DataFrame(raw_data['file_rapor_siswa'])
    # Logic: Keterangan mapping.md: cari id_rapor_siswa di rapor_siswa (db_new)
    # Kita asumsikan idsiswa di file_rapor_siswa (old) memetakan ke idsiswa di rapor (old)
    # dan kita butuh idrapor (old) as id_rapor_siswa (new).
    
    # Karena di rapor_siswa kita simpan id_rapor_siswa yang sama dengan idrapor lama,
    # kita bisa melakukan lookup. 
    # Namun mapping.md bilang: idsiswa (old) -> id_rapor_siswa (new).
    # Mari kita buat mapping dari idsiswa lama ke idrapor lama.
    rapor_map = dict(zip(pd.DataFrame(raw_data['rapor'])['idsiswa'], pd.DataFrame(raw_data['rapor'])['idrapor']))
    
    df['id_rapor_siswa'] = df['idsiswa'].map(rapor_map)
    
    mapping = {
        'idfile': 'id_rapor_siswa_file', 'id_rapor_siswa': 'id_rapor_siswa', 'path': 'file_rapor_path'
    }
    transformed_dfs['rapor_siswa_file'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 9. history_rapor -> rapor_lacak
if 'history_rapor' in raw_data and 'rapor_siswa_file' in transformed_dfs:
    df = pd.DataFrame(raw_data['history_rapor'])
    # Enum normalization
    df['status'] = df['status'].replace({'Terkirim': 'Terkirim', 'Gagal': 'Gagal'})
    
    # id_rapor_siswa_file: cari di rapor_siswa_file
    # Kita butuh mapping dari idsiswa (old) ke id_rapor_siswa_file (new)
    file_map = dict(zip(pd.DataFrame(raw_data['file_rapor_siswa'])['idsiswa'], transformed_dfs['rapor_siswa_file']['id_rapor_siswa_file']))
    
    mapping = {
        'idhistori': 'id_rapor_lacak', 'idsiswa': 'id_siswa',
        'idjadwal': 'id_jadwal', 'tgl': 'tanggal_terkirim', 'status': 'status_pengiriman'
    }
    df_final = df.rename(columns=mapping)
    df_final['id_rapor_siswa_file'] = df_final['id_siswa'].map(file_map)
    
    transformed_dfs['rapor_lacak'] = df_final[list(mapping.values()) + ['id_rapor_siswa_file']]

print(f"✓ Transformasi {len(transformed_dfs)} tabel Fase 5 selesai.")

✓ Transformasi 9 tabel Fase 5 selesai.


## 3.1 Verifikasi Hasil Transformasi
Bagian ini menampilkan perbandingan jumlah data dan tipe data untuk pengecekan manual.

In [5]:
# 3.1.1 Ringkasan Jumlah Baris
print("📊 RINGKASAN MIGRASI (RECORDS COUNT)")
print("="*70)
summary_list = []
for old_t, new_t in hanif_tables_map:
    old_c = len(raw_data.get(old_t, []))
    new_c = len(transformed_dfs.get(new_t, []))
    summary_list.append({
        'Tabel Lama': old_t,
        'Tabel Baru': new_t,
        'Old Recs': old_c,
        'New Recs': new_c,
        'Diff': new_c - old_c,
        'Status': "✅ OK" if old_c == new_c else "⚠️ Cek"
    })
display(pd.DataFrame(summary_list))

total_old_all = sum(len(records) for records in raw_data.values())
total_new_all = sum(len(df) for df in transformed_dfs.values())
print(f"\n📢 TOTAL REKAPITULASI: {total_old_all} (Old) ➔ {total_new_all} (New)")
if total_old_all == total_new_all: print("✅ SEMUA DATA TERANGKUT")
else: print(f"⚠️ ADA SELISIH: {total_new_all - total_old_all} baris")

📊 RINGKASAN MIGRASI (RECORDS COUNT)


,Tabel Lama,Tabel Baru,Old Recs,New Recs,Diff,Status
0,format_rapor,rapor_format,45,45,0,✅ OK
1,format_rapor_detil,rapor_format_sub,129,129,0,✅ OK
2,format_rapor_rumus,rapor_format_formula,3,3,0,✅ OK
3,format_rapor_detil_rumus,rapor_format_formula_sub,1650,1650,0,✅ OK
4,format_raport_level,rapor_level_config,348,348,0,✅ OK
5,rapor,rapor_siswa,22837,22837,0,✅ OK
6,file_rapor_siswa,rapor_siswa_file,1506,1506,0,✅ OK
7,history_rapor,rapor_lacak,1366,1366,0,✅ OK



📢 TOTAL REKAPITULASI: 27884 (Old) ➔ 27884 (New)
✅ SEMUA DATA TERANGKUT


In [6]:
# 3.1.2 Output Pengecekan Kolom Spesifik (KETERANGAN mapping.md)
print("\n🔍 PENGECEKAN TRANSFORMASI SPESIFIK (KETERANGAN mapping.md)")
print("="*70)

# 1. Pengecekan Tanggal Input (Rapor Siswa)
if 'rapor_siswa' in transformed_dfs:
    print("\n[RAPOR_SISWA] Pengecekan tanggal_input (Direct Mapping):")
    display(transformed_dfs['rapor_siswa'][['id_rapor_siswa', 'tanggal_input']].head(5))

# 2. Pengecekan Rapor Lacak (Enum Status)
if 'rapor_lacak' in transformed_dfs:
    print("\n[RAPOR_LACAK] Pengecekan Normalisasi Status Pengiriman:")
    display(transformed_dfs['rapor_lacak']['status_pengiriman'].value_counts())

# 3. Pengecekan Rapor Siswa File
if 'rapor_siswa_file' in transformed_dfs:
    print("\n[RAPOR_SISWA_FILE] Pengecekan path file:")
    display(transformed_dfs['rapor_siswa_file'][['id_rapor_siswa', 'file_rapor_path']].head(5))


🔍 PENGECEKAN TRANSFORMASI SPESIFIK (KETERANGAN mapping.md)

[RAPOR_SISWA] Pengecekan tanggal_input (Direct Mapping):


,id_rapor_siswa,tanggal_input
0,R000154,2023-09-29 15:01:39
1,R000155,2023-09-29 15:01:39
2,R000156,2023-09-29 15:01:39
3,R000157,2023-09-29 15:01:39
4,R000158,2023-09-29 15:01:39



[RAPOR_LACAK] Pengecekan Normalisasi Status Pengiriman:


status_pengiriman
Terkirim    1366
Name: count, dtype: int64


[RAPOR_SISWA_FILE] Pengecekan path file:


,id_rapor_siswa,file_rapor_path
0,R012190,uploads/rapor/S0000329.jpeg
1,R011354,uploads/rapor/S0000474.jpeg
2,R013009,uploads/rapor/S0000481.jpeg
3,R011360,uploads/rapor/S0000475.jpeg
4,R011372,uploads/rapor/S0000482.jpeg


In [7]:
# 3.1.3 Detail Perbandingan Kolom & Tipe Data (Side-by-Side)
print("\n🔍 PERBANDINGAN TIPE DATA SIDE-BY-SIDE")
for old_t, new_t in hanif_tables_map:
    print(f"\n{'='*15} {old_t.upper()} ➔ {new_t.upper()} {'='*15}")
    
    df_old = pd.DataFrame(raw_data.get(old_t, []))
    df_new = transformed_dfs.get(new_t, pd.DataFrame())
    
    if not df_new.empty or not df_old.empty:
        comparison = []
        table_mapping = {}
        if old_t == 'format_rapor': table_mapping = {'idformat_rapor': 'id_rapor_format', 'idpendkursus': 'id_kursus', 'title': 'judul_rapor'}
        elif old_t == 'format_rapor_detil': table_mapping = {'idformat_rd': 'id_rapor_format_sub', 'idformat_rapor': 'id_rapor_format', 'subtitle': 'sub_judul_rapor'}
        elif old_t == 'format_rapor_rumus': table_mapping = {'idfrr': 'id_rapor_format_formula', 'idformat_rapor': 'id_rapor_format', 'param_operator': 'logika_operator'}
        elif old_t == 'format_rapor_detil_rumus': table_mapping = {'idfrdr': 'id_rapor_format_formula_sub', 'idformat_rd': 'id_rapor_format_sub', 'param_operator': 'logika_operator', 'idlevel': 'id_level'}
        elif old_t == 'format_raport_level': table_mapping = {'idformat_rl': 'id_rapor_level_config', 'idlevel': 'id_level', 'idpendkursus': 'id_kursus', 'idformat_rapor': 'id_rapor_format'}
        elif old_t == 'rapor': table_mapping = {'idrapor': 'id_rapor_siswa', 'idjadwal': 'id_jadwal', 'idsiswa': 'id_siswa', 'tanggal': 'tanggal_input', 'idp_nilai': 'id_parameter_nilai', 'nilai': 'final_result'}
        elif old_t == 'file_rapor_siswa': table_mapping = {'idfile': 'id_rapor_siswa_file', 'idsiswa': 'id_rapor_siswa', 'path': 'file_rapor_path'}
        elif old_t == 'history_rapor': table_mapping = {'idhistori': 'id_rapor_lacak', 'idsiswa': 'id_siswa', 'idjadwal': 'id_jadwal', 'tgl': 'tanggal_terkirim', 'status': 'status_pengiriman'}

        for old_col, new_col in table_mapping.items():
            comparison.append({
                'Old Column': old_col,
                'Old Type': str(df_old[old_col].dtype) if not df_old.empty and old_col in df_old.columns else "N/A",
                '➔': '➔',
                'New Column': new_col,
                'New Type': str(df_new[new_col].dtype) if not df_new.empty and new_col in df_new.columns else "N/A"
            })
        
        # Cek kolom baru
        if not df_new.empty:
            for col in df_new.columns:
                if col not in table_mapping.values():
                    comparison.append({
                        'Old Column': '(KOLOM BARU / CUSTOM)',
                        'Old Type': '-',
                        '➔': '➔',
                        'New Column': col,
                        'New Type': str(df_new[col].dtype)
                    })
        
        display(pd.DataFrame(comparison))
        if not df_new.empty:
            print(f"\n--- SAMPLE DATA NEW (2 Baris) ---")
            display(df_new.head(2))
    else:
        print(f"⚠️ Tabel {new_t} kosong.")


🔍 PERBANDINGAN TIPE DATA SIDE-BY-SIDE

=============== FORMAT_RAPOR ➔ RAPOR_FORMAT ===============


,Old Column,Old Type,➔,New Column,New Type
0,idformat_rapor,object,➔,id_rapor_format,object
1,idpendkursus,object,➔,id_kursus,object
2,title,object,➔,judul_rapor,object



--- SAMPLE DATA NEW (2 Baris) ---


,id_rapor_format,id_kursus,judul_rapor
0,F00001,K00001,CLASSROOM ASSESSMENT
1,F00002,K00001,END OF TERM TEST



=============== FORMAT_RAPOR_DETIL ➔ RAPOR_FORMAT_SUB ===============


...

,id_rapor_siswa_file,id_rapor_siswa,file_rapor_path
0,F00006,R012190,uploads/rapor/S0000329.jpeg
1,F00007,R011354,uploads/rapor/S0000474.jpeg



=============== HISTORY_RAPOR ➔ RAPOR_LACAK ===============


,Old Column,Old Type,➔,New Column,New Type
0,idhistori,object,➔,id_rapor_lacak,object
1,idsiswa,object,➔,id_siswa,object
2,idjadwal,object,➔,id_jadwal,object
3,tgl,datetime64[ns],➔,tanggal_terkirim,datetime64[ns]
4,status,object,➔,status_pengiriman,object
5,(KOLOM BARU / CUSTOM),-,➔,id_rapor_siswa_file,object



--- SAMPLE DATA NEW (2 Baris) ---


,id_rapor_lacak,id_siswa,id_jadwal,tanggal_terkirim,status_pengiriman,id_rapor_siswa_file
0,H00001,S0000609,J000000173,2024-11-18 10:57:59,Terkirim,F02273
1,H00005,S0000144,J000000352,2024-12-02 11:37:11,Terkirim,F02288


## 4. Export ke Pickle

In [8]:
file_name = 'fase_5_hanif.pkl'
with open(file_name, 'wb') as f:
    pickle.dump(transformed_dfs, f)

total_records_new = sum(len(df) for df in transformed_dfs.values())
total_records_old = sum(len(records) for records in raw_data.values())

migration_result = {
    'fase': 'fase_5',
    'script': 'script_hanif',
    'fase_num': 5,
    'status': 'ready_for_insert',
    'old_records_total': total_records_old,
    'new_records_total': total_records_new,
    'diff': total_records_new - total_records_old,
    'pickle_file': file_name,
    'timestamp': datetime.now().isoformat()
}
print(json.dumps(migration_result, indent=2))

cursor_old.close()
cursor_new.close()
db_old.close()
db_new.close()

{
  "fase": "fase_5",
  "script": "script_hanif",
  "fase_num": 5,
  "status": "ready_for_insert",
  "old_records_total": 27884,
  "new_records_total": 27884,
  "diff": 0,
  "pickle_file": "fase_5_hanif.pkl",
  "timestamp": "2026-05-11T15:35:00.349705"
}
